# Adım 7: Görselleştirme ve Dashboard

Proje sonuçlarını kapsamlı görseller ve interaktif dashboard ile sunar.

| Bölüm | İçerik |
|-------|--------|
| 1-2 | Kurulum & veri yükleme |
| 3 | Model performans karşılaştırma (grouped bar) |
| 4 | Feature Importance (horizontal bar) |
| 5 | Zaman serisi trend grafikleri |
| 6 | Veri dağılım grafikleri (histogram, pie) |
| 7 | EDA ek görseller |
| 8 | Regresyon: Gerçek vs Tahmin + Residual |
| 9 | Streamlit dashboard başlatma |

## 1. Kurulum

In [ ]:
import subprocess, sys
for pkg in ["pandas", "numpy", "matplotlib", "seaborn",
            "scikit-learn", "pyarrow==15.0.2", "streamlit"]:
    r = subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"],
                       capture_output=True, text=True)
    print(f"{'✅' if r.returncode==0 else '❌'} {pkg}")

## 2. Veri & Model Yükleme

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder
import pyarrow.parquet as pq

warnings.filterwarnings("ignore")
sns.set_theme(style="darkgrid")
plt.rcParams.update({"figure.dpi": 110, "font.size": 11})

PROJECT_ROOT = Path(os.getcwd()).parent
PLOTS_DIR    = PROJECT_ROOT / "data" / "dashboard_plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# --- Veri yükle ---
SAMPLE = 20_000
paths = [
    PROJECT_ROOT / "data/delta/gold/engineered_features/engineered_features.parquet",
    PROJECT_ROOT / "data/raw/yellow_tripdata_2023-01.parquet",
]
df = None
for p in paths:
    if p.exists():
        df = pd.read_parquet(p, engine="pyarrow")
        print(f"✅ Yüklendi: {p.name}  ({len(df):,} satır)")
        break

if df is None:
    raise FileNotFoundError("Veri bulunamadı!")

# Temel sütunları hazırla
if "tpep_pickup_datetime" in df.columns:
    df["pickup_datetime"]     = pd.to_datetime(df["tpep_pickup_datetime"], errors="coerce")
    df["pickup_hour"]         = df["pickup_datetime"].dt.hour
    df["pickup_day_of_week"]  = df["pickup_datetime"].dt.dayofweek + 1
    df["pickup_date"]         = df["pickup_datetime"].dt.date
    df["trip_duration_minutes"] = (
        pd.to_datetime(df["tpep_dropoff_datetime"], errors="coerce") -
        df["pickup_datetime"]
    ).dt.total_seconds() / 60
    df = df[(df["fare_amount"] > 0) & (df["fare_amount"] <= 200) &
            (df["trip_distance"] > 0) & (df["trip_distance"] <= 50)].copy()

label_col = "label" if "label" in df.columns else "fare_amount"

if len(df) > SAMPLE:
    df = df.sample(SAMPLE, random_state=42).reset_index(drop=True)

print(f"Kullanılan satır: {len(df):,}  |  label: '{label_col}'")

In [ ]:
# Modelleri (yeniden) eğit — grafik üretimi için
feature_cols = [c for c in [
    "trip_distance", "passenger_count", "pickup_hour",
    "pickup_day_of_week", "PULocationID", "DOLocationID",
    "is_rush_hour", "is_weekend", "fare_per_mile",
    "cross_borough", "is_airport_trip"
] if c in df.columns]

X = df[feature_cols].fillna(0)
y = df[label_col]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

MODELS = {
    "Linear Regression":      LinearRegression(),
    "Ridge Regression":       Ridge(alpha=1.0),
    "Decision Tree":          DecisionTreeRegressor(max_depth=10, random_state=42),
    "Random Forest":          RandomForestRegressor(n_estimators=50, max_depth=10,
                                                    random_state=42, n_jobs=-1),
    "Gradient Boosted Trees": GradientBoostingRegressor(n_estimators=50, random_state=42),
}

results, fitted = [], {}
for name, m in MODELS.items():
    m.fit(X_train, y_train)
    yp = m.predict(X_test)
    fitted[name] = (m, yp)
    results.append({"Model": name,
                    "RMSE": round(np.sqrt(mean_squared_error(y_test, yp)), 3),
                    "MAE":  round(mean_absolute_error(y_test, yp), 3),
                    "R²":   round(r2_score(y_test, yp), 4)})

res_df = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)
print(res_df.to_string(index=False))

## 3. Model Performans Karşılaştırma (Grouped Bar Chart)

In [ ]:
uzun = res_df.melt(id_vars="Model", value_vars=["RMSE","MAE","R²"],
                   var_name="Metrik", value_name="Değer")

fig, ax = plt.subplots(figsize=(13, 5))
sns.barplot(data=uzun, x="Model", y="Değer", hue="Metrik",
            palette="viridis", ax=ax)
ax.set_title("5 Modelin Performans Karşılaştırması (Grouped Bar Chart)",
             fontsize=13, fontweight="bold")
ax.set_xlabel("")
plt.xticks(rotation=15, ha="right")
ax.legend(loc="upper right")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "01_model_comparison.png", bbox_inches="tight")
plt.show()

## 4. Feature Importance (Horizontal Bar Chart)

In [ ]:
en_iyi_adi = res_df.iloc[0]["Model"]
en_iyi_mdl = fitted[en_iyi_adi][0]

fig, ax = plt.subplots(figsize=(9, 5))
if hasattr(en_iyi_mdl, "feature_importances_"):
    fi = pd.Series(en_iyi_mdl.feature_importances_, index=feature_cols).sort_values()
    colors = ["#e74c3c" if v == fi.max() else "#3498db" for v in fi]
    fi.plot.barh(ax=ax, color=colors, alpha=0.85)
    ax.set_xlabel("Önem Skoru")
else:
    fi = pd.Series(np.abs(en_iyi_mdl.coef_), index=feature_cols).sort_values()
    fi.plot.barh(ax=ax, color="#9b59b6", alpha=0.85)
    ax.set_xlabel("|Katsayı|")

ax.set_title(f"Feature Importance — {en_iyi_adi}", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "02_feature_importance.png", bbox_inches="tight")
plt.show()

## 5. Zaman Serisi Trend Grafikleri (Line Chart)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Saatlik ücret trendi
if "pickup_hour" in df.columns:
    saat = df.groupby("pickup_hour")[label_col].mean()
    axes[0].plot(saat.index, saat.values, marker="o", color="#e74c3c", linewidth=2)
    axes[0].fill_between(saat.index, saat.values, alpha=0.15, color="#e74c3c")
    axes[0].set_title("Saatlik Ortalama Ücret Trendi", fontweight="bold")
    axes[0].set_xlabel("Saat")
    axes[0].set_ylabel("Ortalama Ücret ($)")
    axes[0].set_xticks(range(0, 24, 2))

# Haftanın günü trendi
if "pickup_day_of_week" in df.columns:
    gun = {1:"Paz",2:"Pzt",3:"Sal",4:"Çar",5:"Per",6:"Cum",7:"Cmt"}
    gun_df = df.groupby("pickup_day_of_week")[label_col].mean()
    gun_df.index = gun_df.index.map(gun)
    axes[1].plot(gun_df.index, gun_df.values, marker="s", color="#3498db",
                 linewidth=2, markersize=8)
    axes[1].fill_between(range(len(gun_df)), gun_df.values, alpha=0.15, color="#3498db")
    axes[1].set_xticks(range(len(gun_df)))
    axes[1].set_xticklabels(gun_df.index)
    axes[1].set_title("Haftanın Günü Ortalama Ücret Trendi", fontweight="bold")
    axes[1].set_xlabel("Gün")
    axes[1].set_ylabel("Ortalama Ücret ($)")

plt.suptitle("Zaman Serisi Trend Grafikleri", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "03_time_trends.png", bbox_inches="tight")
plt.show()

## 6. Veri Dağılım Grafikleri (Histogram & Pie Chart)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Histogram — ücret dağılımı
veri = df[label_col].clip(upper=80)
axes[0].hist(veri, bins=60, color="#1abc9c", alpha=0.85, edgecolor="white")
axes[0].axvline(veri.mean(),   color="red",    linestyle="--", lw=2, label=f"Ort: ${veri.mean():.2f}")
axes[0].axvline(veri.median(), color="orange", linestyle="--", lw=2, label=f"Med: ${veri.median():.2f}")
axes[0].set_title("Ücret Dağılımı (Histogram)", fontweight="bold")
axes[0].set_xlabel("Ücret ($)")
axes[0].set_ylabel("Frekans")
axes[0].legend()

# Pie chart — mesafe kategorisi
if "distance_bin" in df.columns:
    pie_data = df["distance_bin"].astype(str).value_counts()
elif "trip_distance" in df.columns:
    labels = ["<1 mil", "1-3 mil", "3-7 mil", "7+ mil"]
    bins   = pd.cut(df["trip_distance"], [0,1,3,7,100], labels=labels)
    pie_data = bins.value_counts()
else:
    pie_data = pd.Series({"Veri yok": 1})

axes[1].pie(pie_data.values, labels=pie_data.index, autopct="%1.1f%%",
            colors=sns.color_palette("husl", len(pie_data)), startangle=90)
axes[1].set_title("Mesafe Kategorisi Dağılımı (Pie Chart)", fontweight="bold")

plt.suptitle("Veri Dağılım Grafikleri", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "04_distributions.png", bbox_inches="tight")
plt.show()

## 7. EDA — 3 Ek Görselleştirme

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 7a: Ücret vs Mesafe scatter
x_col = "trip_distance"
if x_col in df.columns:
    sample = df.sample(min(3000, len(df)), random_state=1)
    axes[0].scatter(sample[x_col], sample[label_col], alpha=0.2, s=8, color="#9b59b6")
    axes[0].set_xlabel("Mesafe (mil)")
    axes[0].set_ylabel("Ücret ($)")
    axes[0].set_title("Ücret vs Mesafe", fontweight="bold")
    axes[0].set_xlim(0, 30)
    axes[0].set_ylim(0, 100)

# 7b: Saat × Gün heatmap
if "pickup_hour" in df.columns and "pickup_day_of_week" in df.columns:
    pivot = df.pivot_table(index="pickup_day_of_week", columns="pickup_hour",
                           values=label_col, aggfunc="mean")
    sns.heatmap(pivot, ax=axes[1], cmap="YlOrRd", linewidths=0.3, cbar_kws={"label": "Ort. Ücret ($)"})
    axes[1].set_title("Gün × Saat Ortalama Ücret Heatmap", fontweight="bold")
    axes[1].set_xlabel("Saat")
    axes[1].set_ylabel("Haftanın Günü")

# 7c: Kutu grafiği — saate göre ücret
if "pickup_hour" in df.columns:
    secilen_saatler = df[df["pickup_hour"].isin([0,6,8,12,17,18,22])]
    sns.boxplot(data=secilen_saatler, x="pickup_hour", y=label_col,
                palette="husl", ax=axes[2], showfliers=False)
    axes[2].set_ylim(0, 60)
    axes[2].set_title("Seçili Saatlerde Ücret Dağılımı (Box Plot)", fontweight="bold")
    axes[2].set_xlabel("Saat")
    axes[2].set_ylabel("Ücret ($)")

plt.suptitle("EDA — Ek Görselleştirmeler", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "05_eda_extra.png", bbox_inches="tight")
plt.show()

## 8. Regresyon: Gerçek vs Tahmin + Residual

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

y_pred_best = fitted[en_iyi_adi][1]
residuals   = y_test.values - y_pred_best

# Gerçek vs Tahmin
axes[0].scatter(y_test, y_pred_best, alpha=0.25, s=10, color="#3498db")
mn, mx = float(y_test.min()), float(y_test.max())
axes[0].plot([mn, mx], [mn, mx], "r--", lw=2, label="İdeal")
axes[0].set_xlabel("Gerçek Ücret ($)")
axes[0].set_ylabel("Tahmin Edilen Ücret ($)")
axes[0].set_title(f"Gerçek vs Tahmin — {en_iyi_adi}", fontweight="bold")
r2 = res_df[res_df["Model"]==en_iyi_adi]["R²"].values[0]
axes[0].legend()
axes[0].text(0.05, 0.92, f"R²={r2}", transform=axes[0].transAxes,
             fontsize=11, color="darkgreen")

# Residual dağılımı
sns.histplot(residuals, bins=60, kde=True, color="#e67e22", ax=axes[1])
axes[1].axvline(0, color="red", lw=2)
axes[1].set_xlabel("Hata ($)")
axes[1].set_ylabel("Frekans")
axes[1].set_title(f"Residual (Artık) Dağılımı — {en_iyi_adi}", fontweight="bold")

plt.suptitle("Regresyon Analiz Grafikleri", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "06_regression_plots.png", bbox_inches="tight")
plt.show()

print(f"Ortalama hata: {residuals.mean():.3f}$  |  Std: {residuals.std():.3f}$")

## 9. Dashboard Grafik Özeti

In [ ]:
grafik_dosyalari = sorted(PLOTS_DIR.glob("*.png"))
print(f"Toplam {len(grafik_dosyalari)} grafik üretildi:")
print("-" * 45)
for g in grafik_dosyalari:
    boyut = g.stat().st_size / 1024
    print(f"  ✅ {g.name:<35} {boyut:.1f} KB")
print(f"\nKonum: {PLOTS_DIR}")

## 10. Streamlit Dashboard Başlatma

Mevcut `ml_pipeline/dashboard.py` dosyasını çalıştırır:

In [ ]:
dashboard_path = PROJECT_ROOT / "ml_pipeline" / "dashboard.py"
print(f"Dashboard dosyası: {dashboard_path}")
print(f"Mevcut: {'✅' if dashboard_path.exists() else '❌'}")
print()
print("Başlatmak için terminalde çalıştırın:")
print(f"  cd {PROJECT_ROOT}")
print(f"  streamlit run ml_pipeline/dashboard.py")
print()
print("  → http://localhost:8501")

## 11. Özet & Teslim Edilecekler

| # | Görsel | Dosya | Durum |
|---|--------|-------|-------|
| 1 | Model performans (grouped bar) | `01_model_comparison.png` | ✅ |
| 2 | Feature Importance (horizontal bar) | `02_feature_importance.png` | ✅ |
| 3 | Zaman serisi trendleri (line chart) | `03_time_trends.png` | ✅ |
| 4 | Veri dağılımı (histogram + pie) | `04_distributions.png` | ✅ |
| 5 | EDA ek görseller (scatter, heatmap, box) | `05_eda_extra.png` | ✅ |
| 6 | Gerçek vs Tahmin + Residual | `06_regression_plots.png` | ✅ |
| 7 | Streamlit dashboard | `ml_pipeline/dashboard.py` | ✅ |

**Grafik çıktıları:** `data/dashboard_plots/`

---
> 🏁 **Tüm adımlar tamamlandı!**  
> `Kafka → Bronze → Silver → Gold → Feature Engineering → ML → Dashboard`